# Youth Opportunity Index Poster Visuals

This notebook generates poster-ready figures and tables for the scholarly poster:

**Mapping Youth Opportunity Deserts in San Diego County**

Sections covered:
- Materials & Methods
- Results
- Conclusion

Outputs are saved to `poster_figures/`.

In [49]:
from pathlib import Path
import warnings
import textwrap

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.cm import ScalarMappable

warnings.filterwarnings("ignore")

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "font.size": 11,
    "axes.titlesize": 14,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# Dashboard-inspired warm-to-cool palette
YOI_CMAP = LinearSegmentedColormap.from_list(
    "yoi_cmap",
    ["#B6442C", "#E59A22", "#EEC574", "#7CC6BB", "#2B989E", "#246E7E", "#27373D"]
)

DOMAIN_LABELS = {
    "economic_score": "Economic",
    "education_score": "Education",
    "health_score": "Health",
    "housing_score": "Housing",
    "safety_env_score": "Safety / Env",
    "mobility_connectivity_score": "Mobility / Connectivity",
    "youth_supports_score": "Youth Supports",
}

DOMAIN_COLS = list(DOMAIN_LABELS.keys())

In [50]:
def find_repo_root(start: Path) -> Path:
    for p in [start] + list(start.parents):
        if (p / "data").exists():
            return p
    raise FileNotFoundError("Could not find repo root (directory containing /data).")

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = find_repo_root(NOTEBOOK_DIR)

OUT = NOTEBOOK_DIR / "poster_figures"
OUT.mkdir(exist_ok=True)

PATHS = {
    "yoi": REPO_ROOT / "data" / "processed" / "yoi" / "yoi_components.csv",
    "indicator_meta": REPO_ROOT / "data" / "processed" / "yoi" / "yoi_indicator_meta.csv",
    "zip": REPO_ROOT / "data" / "processed" / "yoi" / "yoi_zip_components.csv",
    "supervisor": REPO_ROOT / "data" / "processed" / "yoi" / "yoi_supervisor_district_components.csv",
    "tracts": REPO_ROOT / "data" / "processed" / "boundaries" / "sd_tracts.geojson",
    "coi": REPO_ROOT / "data" / "processed" / "overlays" / "sd_coi_2023.csv",
    "services": REPO_ROOT / "data" / "processed" / "overlays" / "service_locations.geojson",
    "routes": REPO_ROOT / "data" / "processed" / "boundaries" / "transit_routes.geojson",
    "stops": REPO_ROOT / "data" / "processed" / "boundaries" / "transit_stops.geojson",
}

for k, v in PATHS.items():
    print(f"{k:>12}: {v.exists()}  {v}")

         yoi: True  /Users/laurenvo/Documents/github/youth_opportunity_index/data/processed/yoi/yoi_components.csv
indicator_meta: True  /Users/laurenvo/Documents/github/youth_opportunity_index/data/processed/yoi/yoi_indicator_meta.csv
         zip: True  /Users/laurenvo/Documents/github/youth_opportunity_index/data/processed/yoi/yoi_zip_components.csv
  supervisor: True  /Users/laurenvo/Documents/github/youth_opportunity_index/data/processed/yoi/yoi_supervisor_district_components.csv
      tracts: True  /Users/laurenvo/Documents/github/youth_opportunity_index/data/processed/boundaries/sd_tracts.geojson
         coi: True  /Users/laurenvo/Documents/github/youth_opportunity_index/data/processed/overlays/sd_coi_2023.csv
    services: True  /Users/laurenvo/Documents/github/youth_opportunity_index/data/processed/overlays/service_locations.geojson
      routes: True  /Users/laurenvo/Documents/github/youth_opportunity_index/data/processed/boundaries/transit_routes.geojson
       stops: True 

In [51]:
def normalize_geoid(v):
    if pd.isna(v):
        return None
    s = "".join(ch for ch in str(v) if ch.isdigit())
    return s.zfill(11)[-11:] if s else None

def tract_label_from_geoid(geoid):
    g = normalize_geoid(geoid)
    if not g:
        return "Census tract"
    suffix = g[5:]
    return f"{int(suffix[:4])}.{suffix[4:]}"

def load_csv_if_exists(path, **kwargs):
    return pd.read_csv(path, **kwargs) if path.exists() else None

def load_gdf_if_exists(path):
    return gpd.read_file(path) if path.exists() else None

def savefig(fig, filename):
    path = OUT / filename
    fig.savefig(path, dpi=300, bbox_inches="tight")
    print(f"Saved {path}")
    plt.close(fig)

def first_existing_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def score_series_to_100(series):
    if series is None:
        return None
    s = pd.to_numeric(series, errors="coerce")
    return s * 100 if s.max(skipna=True) <= 1.5 else s

def wrap_text(s, width=34):
    return "\n".join(textwrap.wrap(str(s), width=width))

def zscore(series):
    s = pd.to_numeric(series, errors="coerce")
    return (s - s.mean()) / s.std() if s.std() not in [0, np.nan] else s * 0

In [52]:
if not PATHS["yoi"].exists():
    raise FileNotFoundError("Missing data/processed/yoi/yoi_components.csv")

if not PATHS["tracts"].exists():
    raise FileNotFoundError("Missing data/processed/boundaries/sd_tracts.geojson")

yoi = pd.read_csv(PATHS["yoi"], dtype={"tract_geoid": str})
yoi["tract_geoid"] = yoi["tract_geoid"].map(normalize_geoid)

indicator_meta = load_csv_if_exists(PATHS["indicator_meta"])
zip_df = load_csv_if_exists(PATHS["zip"])
sup_df = load_csv_if_exists(PATHS["supervisor"])
coi = load_csv_if_exists(PATHS["coi"], dtype={"tract_geoid": str})
services = load_gdf_if_exists(PATHS["services"])
routes = load_gdf_if_exists(PATHS["routes"])
stops = load_gdf_if_exists(PATHS["stops"])

tracts = gpd.read_file(PATHS["tracts"])
tract_geoid_col = first_existing_col(tracts, ["tract_geoid", "GEOID", "geoid"])
tracts[tract_geoid_col] = tracts[tract_geoid_col].astype(str).map(normalize_geoid)
tracts = tracts.rename(columns={tract_geoid_col: "tract_geoid"})

if coi is not None:
    coi["tract_geoid"] = coi["tract_geoid"].map(normalize_geoid)

OVERALL_COL = first_existing_col(yoi, ["yoi_custom_0_100", "yoi_0_100", "yoi_raw_0_1"])
POP_COL = first_existing_col(yoi, ["total_population", "population", "pop_total"])
SERVICES_PER_10K_COL = first_existing_col(yoi, ["youth_services_per_10k", "services_per_10k", "mh_services_per_10k"])

# Clean display scores
yoi["overall_yoi_100"] = score_series_to_100(yoi[OVERALL_COL])

for c in DOMAIN_COLS:
    if c in yoi.columns:
        yoi[f"{c}_100"] = score_series_to_100(yoi[c])

gdf = tracts.merge(yoi, on="tract_geoid", how="left")

if coi is not None and "coi_score" in coi.columns:
    coi["coi_score"] = pd.to_numeric(coi["coi_score"], errors="coerce")
    gdf = gdf.merge(coi[["tract_geoid", "coi_score"]], on="tract_geoid", how="left")

# Domain-based helpers
domain_medians = {c: yoi[f"{c}_100"].median(skipna=True) for c in DOMAIN_COLS if f"{c}_100" in yoi.columns}
yoi["domains_below_median"] = sum((yoi[f"{c}_100"] < domain_medians[c]).fillna(False) for c in DOMAIN_COLS if f"{c}_100" in yoi.columns)
gdf = gdf.drop(columns=["domains_below_median"], errors="ignore").merge(
    yoi[["tract_geoid", "domains_below_median"]], on="tract_geoid", how="left"
)

print("Overall score column:", OVERALL_COL)
print("Population column:", POP_COL)
print("Service-density column:", SERVICES_PER_10K_COL)
print("Rows in yoi:", len(yoi))
print("Rows in tracts:", len(tracts))
display(yoi.head())

Overall score column: yoi_0_100
Population column: total_population
Service-density column: youth_services_per_10k
Rows in yoi: 737
Rows in tracts: 737


,tract_geoid,economic_score,economic_coverage,education_score,education_coverage,health_score,health_coverage,housing_score,housing_coverage,safety_env_score,...,lack_transport,overall_yoi_100,economic_score_100,education_score_100,health_score_100,housing_score_100,safety_env_score_100,mobility_connectivity_score_100,youth_supports_score_100,domains_below_median
0,06073017039,0.573915,1.0,0.749792,1.0,0.598263,1.0,0.527995,1.0,0.675539,...,5.7,67.941798,57.391539,74.979166,59.826309,52.799461,67.553876,69.330415,93.711821,0
1,06073017040,0.586179,1.0,0.505535,1.0,0.577510,1.0,0.672448,1.0,0.721669,...,5.5,60.894925,58.617897,50.553517,57.750971,67.244837,72.166903,71.288955,48.641394,1
2,06073017043,0.770469,1.0,0.581141,1.0,0.628792,1.0,0.546416,1.0,0.777464,...,4.9,69.443377,77.046858,58.114067,62.879249,54.641621,77.746397,67.229369,88.446081,0
3,06073003103,0.500664,1.0,0.478576,1.0,0.184932,1.0,0.649999,1.0,0.702295,...,12.1,48.955667,50.066390,47.857563,18.493248,64.999868,70.229459,17.887489,73.155650,4
4,06073003105,0.393974,1.0,0.515278,1.0,0.369414,1.0,0.500808,1.0,0.763272,...,12.6,54.370473,39.397367,51.527842,36.941406,50.080764,76.327218,35.424577,90.894138,3


## Materials & Methods — Figure M1
### Domain-by-source indicator provenance heatmap

In [53]:
def infer_meta_column(df, keywords):
    if df is None:
        return None
    cols = [c for c in df.columns]
    for kw in keywords:
        for c in cols:
            if kw.lower() in c.lower():
                return c
    return None

if indicator_meta is not None:
    domain_col = infer_meta_column(indicator_meta, ["domain"])
    source_col = infer_meta_column(indicator_meta, ["source", "dataset"])
    if domain_col is not None and source_col is not None:
        meta_plot = indicator_meta.copy()
        meta_plot[domain_col] = meta_plot[domain_col].astype(str)
        meta_plot[source_col] = meta_plot[source_col].astype(str)

        # Clean domain labels if they come in code-like names
        meta_plot["domain_clean"] = meta_plot[domain_col].replace(DOMAIN_LABELS)
        meta_plot["source_clean"] = meta_plot[source_col].str.replace("_", " ").str.strip()

        cross = pd.crosstab(meta_plot["source_clean"], meta_plot["domain_clean"])
        cross = cross.loc[cross.sum(axis=1).sort_values(ascending=False).index]

        fig, ax = plt.subplots(figsize=(10, 5.8))
        im = ax.imshow(cross.values, aspect="auto", cmap="Blues")

        ax.set_xticks(np.arange(len(cross.columns)))
        ax.set_xticklabels(cross.columns, rotation=25, ha="right")
        ax.set_yticks(np.arange(len(cross.index)))
        ax.set_yticklabels(cross.index)

        for i in range(cross.shape[0]):
            for j in range(cross.shape[1]):
                val = int(cross.iloc[i, j])
                ax.text(j, i, val, ha="center", va="center", fontsize=9, color="black")

        cbar = fig.colorbar(im, ax=ax, shrink=0.82, pad=0.02)
        cbar.set_label("Number of indicators")

        ax.set_title("Figure M1. Indicator provenance by domain and source", weight="bold")
        savefig(fig, "methods_M1_indicator_provenance_heatmap.png")
        display(cross)
    else:
        print("Indicator metadata exists, but source/domain columns were not detected.")
else:
    print("Skipping M1: yoi_indicator_meta.csv not found.")

Saved /Users/laurenvo/Documents/github/youth_opportunity_index/notebooks/poster_figures/methods_M1_indicator_provenance_heatmap.png


domain_clean,economic,education,health,housing,mobility_connectivity,safety_env,youth_supports
source_clean,,,,,,,
ACS S1501,0,2,0,0,0,0,0
CDC PLACES,0,0,2,0,0,0,0
ACS B07001,0,0,0,1,0,0,0
services master,0,0,0,0,0,0,1
SANDAG ARJIS CIBRS,0,0,0,0,0,1,0
CalEnviroScreen 4.0,0,0,0,0,0,1,0
ACS S2701,0,0,1,0,0,0,0
ACS S2301,1,0,0,0,0,0,0
ACS S2201,1,0,0,0,0,0,0


## Materials & Methods — Figure/Table M2
### Processed output and overlay inventory table

In [54]:
summary_rows = []

summary_rows.append({
    "Layer": "Tract-level YOI",
    "Count": len(yoi),
    "Role": "Primary analysis unit",
    "Key output": "yoi_components.csv"
})

if zip_df is not None:
    summary_rows.append({
        "Layer": "ZIP-level YOI",
        "Count": len(zip_df),
        "Role": "Broader public-facing geography",
        "Key output": "yoi_zip_components.csv"
    })

if sup_df is not None:
    summary_rows.append({
        "Layer": "Supervisor districts",
        "Count": len(sup_df),
        "Role": "County planning geography",
        "Key output": "yoi_supervisor_district_components.csv"
    })

summary_rows.append({
    "Layer": "Tract boundaries",
    "Count": len(tracts),
    "Role": "Base choropleth geometry",
    "Key output": "sd_tracts.geojson"
})

if coi is not None:
    summary_rows.append({
        "Layer": "COI overlay",
        "Count": len(coi),
        "Role": "External benchmark layer",
        "Key output": "sd_coi_2023.csv"
    })

if services is not None:
    summary_rows.append({
        "Layer": "Service locations",
        "Count": len(services),
        "Role": "Contextual service overlay",
        "Key output": "service_locations.geojson"
    })

if routes is not None:
    summary_rows.append({
        "Layer": "Transit routes",
        "Count": len(routes),
        "Role": "Mobility context overlay",
        "Key output": "transit_routes.geojson"
    })

if stops is not None:
    summary_rows.append({
        "Layer": "Transit stops",
        "Count": len(stops),
        "Role": "Mobility access overlay",
        "Key output": "transit_stops.geojson"
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUT / "methods_M2_output_inventory.csv", index=False)
display(summary_df)

# Render as poster-friendly table figure
fig, ax = plt.subplots(figsize=(11.5, 0.6 * len(summary_df) + 1.6))
ax.axis("off")

table_data = [summary_df.columns.tolist()] + summary_df.values.tolist()
table = ax.table(
    cellText=table_data,
    loc="center",
    cellLoc="left",
    colLoc="left"
)

table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.45)

for (r, c), cell in table.get_celld().items():
    cell.set_edgecolor("#D1D5DB")
    if r == 0:
        cell.set_facecolor("#EAF3FB")
        cell.set_text_props(weight="bold", color="#0F172A")
    else:
        cell.set_facecolor("white")

ax.set_title("Figure M2. Processed outputs and contextual layers used in the dashboard", weight="bold", pad=12)
savefig(fig, "methods_M2_output_inventory_table.png")

,Layer,Count,Role,Key output
0,Tract-level YOI,737,Primary analysis unit,yoi_components.csv
1,ZIP-level YOI,113,Broader public-facing geography,yoi_zip_components.csv
2,Supervisor districts,5,County planning geography,yoi_supervisor_district_components.csv
3,Tract boundaries,737,Base choropleth geometry,sd_tracts.geojson
4,COI overlay,736,External benchmark layer,sd_coi_2023.csv
5,Service locations,5592,Contextual service overlay,service_locations.geojson
6,Transit routes,838,Mobility context overlay,transit_routes.geojson
7,Transit stops,6220,Mobility access overlay,transit_stops.geojson


Saved /Users/laurenvo/Documents/github/youth_opportunity_index/notebooks/poster_figures/methods_M2_output_inventory_table.png


## Results — Figure R1
### Main countywide tract-level YOI map

In [55]:
plot_gdf = gdf.dropna(subset=["overall_yoi_100"]).copy()
bottom5 = plot_gdf.nsmallest(5, "overall_yoi_100").copy()

fig, ax = plt.subplots(figsize=(8.6, 8.6))

plot_gdf.plot(
    column="overall_yoi_100",
    cmap=YOI_CMAP,
    linewidth=0.15,
    edgecolor=(1, 1, 1, 0.18),
    ax=ax,
    vmin=0,
    vmax=100
)

bottom5.boundary.plot(ax=ax, color="black", linewidth=1.1)

for _, row in bottom5.iterrows():
    pt = row.geometry.representative_point()
    ax.text(
        pt.x, pt.y,
        tract_label_from_geoid(row["tract_geoid"]),
        fontsize=8,
        weight="bold",
        ha="center",
        va="center",
        bbox=dict(boxstyle="round,pad=0.12", fc="white", ec="none", alpha=0.9)
    )

sm = ScalarMappable(norm=Normalize(vmin=0, vmax=100), cmap=YOI_CMAP)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, shrink=0.58, pad=0.04)
cbar.set_label("Overall YOI (0–100)")

ax.set_title("Overall Youth Opportunity Index across San Diego County tracts", weight="bold")
ax.set_axis_off()

savefig(fig, "results_R1_overall_yoi_map.png")

Saved /Users/laurenvo/Documents/github/youth_opportunity_index/notebooks/poster_figures/results_R1_overall_yoi_map.png


## Results — Figure R2
### Heatmap of low-opportunity tract typologies

In [56]:
low_df = yoi[yoi["overall_yoi_100"] <= yoi["overall_yoi_100"].quantile(0.25)].copy()

econ_q25 = low_df["economic_score_100"].quantile(0.25) if "economic_score_100" in low_df.columns else np.nan
ys_q25 = low_df["youth_supports_score_100"].quantile(0.25) if "youth_supports_score_100" in low_df.columns else np.nan

def assign_typology(row):
    broad = row["domains_below_median"] >= 5
    econ_weak = row.get("economic_score_100", np.nan) <= econ_q25 if pd.notna(econ_q25) else False
    ys_weak = row.get("youth_supports_score_100", np.nan) <= ys_q25 if pd.notna(ys_q25) else False

    if broad:
        return "Broad multi-domain deficit"
    elif econ_weak and not ys_weak:
        return "Economic-driven"
    elif ys_weak and not econ_weak:
        return "Youth-support-driven"
    else:
        return "Mixed low-opportunity"

low_df["typology"] = low_df.apply(assign_typology, axis=1)

typology_profile = low_df.groupby("typology")[[f"{c}_100" for c in DOMAIN_COLS]].mean()
typology_counts = low_df["typology"].value_counts()

row_order = [
    "Broad multi-domain deficit",
    "Economic-driven",
    "Youth-support-driven",
    "Mixed low-opportunity"
]
row_order = [r for r in row_order if r in typology_profile.index]
typology_profile = typology_profile.loc[row_order]
typology_profile.columns = [DOMAIN_LABELS[c] for c in DOMAIN_COLS]

fig, ax = plt.subplots(figsize=(11.2, 5.2))
im = ax.imshow(typology_profile.values, aspect="auto", cmap="RdYlGn", vmin=0, vmax=100)

ax.set_xticks(np.arange(len(typology_profile.columns)))
ax.set_xticklabels(typology_profile.columns, rotation=25, ha="right")

row_labels = [f"{name} (n={typology_counts[name]})" for name in typology_profile.index]
ax.set_yticks(np.arange(len(row_labels)))
ax.set_yticklabels(row_labels)

for i in range(typology_profile.shape[0]):
    for j in range(typology_profile.shape[1]):
        val = typology_profile.iloc[i, j]
        ax.text(j, i, f"{val:.0f}", ha="center", va="center", fontsize=9, color="black")

cbar = fig.colorbar(im, ax=ax, shrink=0.82, pad=0.02)
cbar.set_label("Average domain score (0–100)")

# ax.set_title("Low-opportunity tracts are not all the same", weight="bold")
savefig(fig, "results_R2_low_opportunity_typology_heatmap.png")

Saved /Users/laurenvo/Documents/github/youth_opportunity_index/notebooks/poster_figures/results_R2_low_opportunity_typology_heatmap.png


## Results — Figure R3
### Opportunity versus youth-service coverage

In [57]:
if SERVICES_PER_10K_COL is None:
    print("Skipping R3: no service-density column found in yoi_components.csv")
else:
    plot_df = yoi.dropna(subset=["overall_yoi_100", SERVICES_PER_10K_COL, POP_COL]).copy() if POP_COL else yoi.dropna(subset=["overall_yoi_100", SERVICES_PER_10K_COL]).copy()
    plot_df["service_display"] = pd.to_numeric(plot_df[SERVICES_PER_10K_COL], errors="coerce")
    plot_df = plot_df.dropna(subset=["service_display"])

    # Trim extreme outliers for readability
    x_cap = plot_df["service_display"].quantile(0.99)
    plot_df["service_display"] = plot_df["service_display"].clip(upper=x_cap)

    fig, ax = plt.subplots(figsize=(9.6, 6.6))

    hb = ax.hexbin(
        plot_df["service_display"],
        plot_df["overall_yoi_100"],
        gridsize=24,
        mincnt=1,
        cmap="Blues",
        linewidths=0.2
    )

    highlight = plot_df.nsmallest(8, "overall_yoi_100").copy()
    ax.scatter(
        highlight["service_display"],
        highlight["overall_yoi_100"],
        s=52,
        facecolors="#D95F02",
        edgecolors="black",
        linewidths=0.7,
        zorder=3
    )

    for _, row in highlight.iterrows():
        ax.annotate(
            tract_label_from_geoid(row["tract_geoid"]),
            (row["service_display"], row["overall_yoi_100"]),
            xytext=(5, 4),
            textcoords="offset points",
            fontsize=8,
            bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.9),
            zorder=4
        )

    x_med = plot_df["service_display"].median()
    y_med = plot_df["overall_yoi_100"].median()
    ax.axvline(x_med, linestyle="--", linewidth=1.0, color="gray", alpha=0.9)
    ax.axhline(y_med, linestyle="--", linewidth=1.0, color="gray", alpha=0.9)

    cbar = plt.colorbar(hb, ax=ax, pad=0.02)
    cbar.set_label("Number of tracts")

    ax.set_xlabel(f"{SERVICES_PER_10K_COL.replace('_', ' ').title()}")
    ax.set_ylabel("Overall YOI (0–100)")
    # ax.set_title("Opportunity versus local service coverage", weight="bold")
    ax.grid(alpha=0.12)
    ax.set_axisbelow(True)
    ax.set_xlim(left=0)
    ax.set_ylim(0, max(100, plot_df["overall_yoi_100"].max() + 2))

    savefig(fig, "results_R3_opportunity_vs_service_hexbin.png")

Saved /Users/laurenvo/Documents/github/youth_opportunity_index/notebooks/poster_figures/results_R3_opportunity_vs_service_hexbin.png


## Results — Figure R4
### Where the local YOI diverges from the external COI benchmark

In [58]:
if "coi_score" not in gdf.columns or gdf["coi_score"].dropna().empty:
    print("Skipping R4: COI overlay not available.")
else:
    compare_df = gdf.dropna(subset=["overall_yoi_100", "coi_score"]).copy()
    compare_df["diff_yoi_minus_coi"] = compare_df["overall_yoi_100"] - compare_df["coi_score"]
    compare_df["abs_diff"] = compare_df["diff_yoi_minus_coi"].abs()

    fig, ax = plt.subplots(figsize=(8.2, 7.2))

    sc = ax.scatter(
        compare_df["coi_score"],
        compare_df["overall_yoi_100"],
        c=compare_df["domains_below_median"],
        cmap="viridis_r",
        alpha=0.7,
        s=28,
        edgecolors="white",
        linewidths=0.25
    )

    ax.plot([0, 100], [0, 100], linestyle="--", color="gray", linewidth=1.2)
    ax.set_xlim(0, 100)
    ax.set_ylim(0, 100)
    ax.set_xlabel("COI score (0–100)")
    ax.set_ylabel("YOI score (0–100)")
    # ax.set_title("Local YOI versus external COI benchmark", weight="bold")

    # Highlight largest divergences
    outliers = compare_df.nlargest(8, "abs_diff")
    for _, row in outliers.iterrows():
        ax.annotate(
            tract_label_from_geoid(row["tract_geoid"]),
            (row["coi_score"], row["overall_yoi_100"]),
            xytext=(5, 4),
            textcoords="offset points",
            fontsize=8,
            bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.9)
        )

    cbar = plt.colorbar(sc, ax=ax, pad=0.02)
    cbar.set_label("Domains below county median")
    ax.grid(alpha=0.15)
    ax.set_axisbelow(True)

    savefig(fig, "results_R4_yoi_vs_coi_scatter.png")

Saved /Users/laurenvo/Documents/github/youth_opportunity_index/notebooks/poster_figures/results_R4_yoi_vs_coi_scatter.png


## Conclusion — Figure C1
### Priority tracts where multiple disadvantages overlap

In [59]:
priority_df = gdf.dropna(subset=["overall_yoi_100", "youth_supports_score_100"]).copy()

priority_terms = [
    0.40 * zscore(100 - priority_df["overall_yoi_100"]),
    0.25 * zscore(100 - priority_df["youth_supports_score_100"]),
]

if SERVICES_PER_10K_COL is not None and SERVICES_PER_10K_COL in priority_df.columns:
    service_numeric = pd.to_numeric(priority_df[SERVICES_PER_10K_COL], errors="coerce")
    priority_terms.append(0.20 * zscore(-service_numeric))

if POP_COL is not None and POP_COL in priority_df.columns:
    pop_numeric = pd.to_numeric(priority_df[POP_COL], errors="coerce")
    priority_terms.append(0.15 * zscore(pop_numeric))

priority_df["priority_score"] = sum(priority_terms)
priority_df = priority_df.dropna(subset=["priority_score"]).sort_values("priority_score", ascending=False)

top_priority = priority_df.head(10).copy()

fig, ax = plt.subplots(figsize=(8.6, 8.6))
priority_df.plot(
    column="priority_score",
    cmap="magma_r",
    linewidth=0.15,
    edgecolor=(1, 1, 1, 0.16),
    ax=ax
)

top_priority.boundary.plot(ax=ax, color="black", linewidth=1.2)

for _, row in top_priority.iterrows():
    pt = row.geometry.representative_point()
    ax.text(
        pt.x, pt.y,
        tract_label_from_geoid(row["tract_geoid"]),
        fontsize=8,
        weight="bold",
        ha="center",
        va="center",
        bbox=dict(boxstyle="round,pad=0.12", fc="white", ec="none", alpha=0.9)
    )

sm = ScalarMappable(norm=Normalize(vmin=priority_df["priority_score"].min(), vmax=priority_df["priority_score"].max()), cmap="magma_r")
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, shrink=0.76, pad=0.02)
cbar.set_label("Priority score")

ax.set_title("Figure C1. Priority tracts where low opportunity and weak supports overlap", weight="bold")
ax.set_axis_off()

savefig(fig, "conclusion_C1_priority_tract_map.png")

Saved /Users/laurenvo/Documents/github/youth_opportunity_index/notebooks/poster_figures/conclusion_C1_priority_tract_map.png


## Conclusion — Table C2
### Top-priority tracts and likely intervention focus

In [60]:
table_df = top_priority.copy()

def weakest_domains(row, n=2):
    vals = {DOMAIN_LABELS[c]: row.get(f"{c}_100", np.nan) for c in DOMAIN_COLS}
    vals = {k: v for k, v in vals.items() if pd.notna(v)}
    ordered = sorted(vals.items(), key=lambda x: x[1])
    return ", ".join([name for name, _ in ordered[:n]])

ranked = yoi[["tract_geoid", "overall_yoi_100"]].dropna().sort_values("overall_yoi_100", ascending=False).reset_index(drop=True)
rank_map = {row["tract_geoid"]: i + 1 for i, row in ranked.iterrows()}
total_n = len(ranked)

table_df["Tract"] = table_df["tract_geoid"].map(tract_label_from_geoid)
table_df["Overall YOI"] = table_df["overall_yoi_100"].round(1)
table_df["County rank"] = table_df["tract_geoid"].map(rank_map)
table_df["Percentile"] = ((1 - (table_df["County rank"] - 1) / total_n) * 100).round(0).astype("Int64")
table_df["Weakest domains"] = table_df.apply(weakest_domains, axis=1)
table_df["Domains below median"] = table_df["domains_below_median"].astype("Int64")

if SERVICES_PER_10K_COL is not None:
    table_df["Service density"] = pd.to_numeric(table_df[SERVICES_PER_10K_COL], errors="coerce").round(1)

cols = ["Tract", "Overall YOI", "County rank", "Percentile", "Weakest domains", "Domains below median"]
if "Service density" in table_df.columns:
    cols.append("Service density")

priority_table = table_df[cols].copy()
priority_table.to_csv(OUT / "conclusion_C2_priority_tract_table.csv", index=False)
display(priority_table)

# Render poster-friendly table figure
fig, ax = plt.subplots(figsize=(13, 0.55 * len(priority_table) + 1.5))
ax.axis("off")

table_data = [priority_table.columns.tolist()] + priority_table.values.tolist()
tbl = ax.table(cellText=table_data, loc="center", cellLoc="left", colLoc="left")
tbl.auto_set_font_size(False)
tbl.set_fontsize(9.5)
tbl.scale(1, 1.35)

for (r, c), cell in tbl.get_celld().items():
    cell.set_edgecolor("#D1D5DB")
    if r == 0:
        cell.set_facecolor("#EAF3FB")
        cell.set_text_props(weight="bold", color="#0F172A")
    else:
        cell.set_facecolor("white")

ax.set_title("Table C2. Highest-priority tracts and dominant opportunity gaps", weight="bold", pad=10)
savefig(fig, "conclusion_C2_priority_tract_table.png")

,Tract,Overall YOI,County rank,Percentile,Weakest domains,Domains below median,Service density
247,187.00,44.0,510,31,"Housing, Economic",4,2.0
122,118.02,24.2,735,0,"Youth Supports, Economic",7,0.0
512,203.08,25.2,732,1,"Youth Supports, Health",7,0.0
50,164.04,20.9,737,0,"Economic, Mobility / Connectivity",7,2.4
69,186.22,26.9,724,2,"Youth Supports, Economic",7,0.0
9,104.02,25.5,731,1,"Youth Supports, Economic",7,1.8
348,206.02,28.8,714,3,"Health, Youth Supports",5,0.0
599,125.01,26.3,728,1,"Youth Supports, Economic",6,0.0
355,159.01,23.5,736,0,"Economic, Housing",7,0.0
584,168.09,30.9,697,6,"Youth Supports, Mobility / Connectivity",5,0.0


Saved /Users/laurenvo/Documents/github/youth_opportunity_index/notebooks/poster_figures/conclusion_C2_priority_tract_table.png
